## Create the list file of Raw Videos

In [13]:
import os
import pickle
import gzip
from pathlib import Path

# === CONFIGURATION ===
input_dir = Path("iSign-videos_v1.1")  # Change this to your raw video directory
output_path = Path("./")
output_path.mkdir(parents=True, exist_ok=True)

output_file = output_path / "raw_files_list.pkl.gz"  # This matches your main code
video_exts = {'.mp4', '.avi', '.mov', '.mkv', '.webm'}

# === SCAN VIDEO FILES ===
video_paths = [
    str(file.resolve()) for file in input_dir.rglob("*")
    if file.suffix.lower() in video_exts
]

print(f"Found {len(video_paths)} video files.")

# === SAVE TO .pkl.gz ===
with gzip.open(output_file, 'wb') as f:
    pickle.dump(video_paths, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Saved video path list to: {output_file}")


Found 127237 video files.
Saved video path list to: raw_files_list.pkl.gz


## Pose detection of Hands and Face

In [14]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import cv2
import numpy as np
import os
import json
import time
from pathlib import Path
import decord
from pdb import set_trace as st

In [15]:
def compute_stats_from_pose_data(pose_path, stats_path, video_file):
    landmark_json_path = Path(f"{pose_path}{video_file.split('/')[-1].rsplit('.', 1)[0]}_pose.json")
    stats_json_path = Path(f"{stats_path}{video_file.split('/')[-1].rsplit('.', 1)[0]}_stats.json")

    with open(landmark_json_path, 'r') as f:
        pose_data = json.load(f)

    stats = {}
    max_counts = {'#face': 0, '#hands': 0, '#pose': 0}

    for frame, landmarks in pose_data.items():
        if landmarks is None:
            presence = {'#face': 0, '#hands': 0, '#pose': 0}
        else:
            presence = {
                '#face': len(landmarks.get('face_landmarks', [])) if landmarks.get('face_landmarks') else 0,
                '#hands': len(landmarks.get('hand_landmarks', [])) if landmarks.get('hand_landmarks') else 0,
                '#pose': len(landmarks.get('pose_landmarks', [])) if landmarks.get('pose_landmarks') else 0
            }
        stats[frame] = presence
        for key in max_counts:
            max_counts[key] = max(max_counts[key], presence[key])

    stats['max'] = max_counts

    with open(stats_json_path, 'w') as f:
        json.dump(stats, f)

In [16]:
def is_string_in_file(file_path, target_string):
    try:
        with Path(file_path).open("r") as f:
            return any(target_string in line for line in f)
    except Exception as e:
        print(f"Error: {e}")
        return False

def resize_frame(frame, frame_size):
    return cv2.resize(frame, frame_size, interpolation=cv2.INTER_AREA)

def crop_frame(image, bounding_box):
    x, y, w, h = bounding_box
    return image[y:y + h, x:x + w]

def process_landmarks(landmarks, image_shape):
    ih, iw, _ = image_shape
    landmarks_px = np.array([(int(l.x * iw), int(l.y * ih)) for l in landmarks])
    x, y, w, h = cv2.boundingRect(landmarks_px)
    scale_factor = 1.2
    w_padding = int((scale_factor - 1) * w / 2)
    h_padding = int((scale_factor - 1) * h / 2)
    return x - w_padding, y - h_padding, w + 2 * w_padding, h + 2 * h_padding


In [17]:
def detect_holistic(image):
    results = mp_holistic.process(image)

    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image)
    face_prediction = face_detector.detect(mp_image)
    hand_prediction = hand_detector.detect(mp_image)

    bounding_boxes = {}
    landmarks_data = {}

    if face_prediction.face_landmarks:
        bounding_boxes['#face'] = len(face_prediction.face_landmarks)
        landmarks_data['face_landmarks'] = [
            [[lm.x, lm.y, lm.z] for lm in face] for face in face_prediction.face_landmarks
        ]
    else:
        bounding_boxes['#face'] = 0
        landmarks_data['face_landmarks'] = None

    if hand_prediction.hand_landmarks:
        bounding_boxes['#hands'] = len(hand_prediction.hand_landmarks)
        landmarks_data['hand_landmarks'] = [
            [[lm.x, lm.y, lm.z] for lm in hand] for hand in hand_prediction.hand_landmarks
        ]
    else:
        bounding_boxes['#hands'] = 0
        landmarks_data['hand_landmarks'] = None

    if results.pose_landmarks:
        bounding_boxes['#pose'] = 1
        landmarks_data['pose_landmarks'] = [
            [[lm.x, lm.y, lm.z] for lm in results.pose_landmarks.landmark]
        ]
    else:
        bounding_boxes['#pose'] = 0
        landmarks_data['pose_landmarks'] = None

    return bounding_boxes, landmarks_data


In [18]:
def video_holistic(video_file, problem_file_path, pose_path, stats_path):
    try:
        video = decord.VideoReader(video_file)
    except Exception as e:
        print(f"Error loading video: {e}")
        with open(problem_file_path, "a") as p:
            p.write(video_file + "\n")
        return

    result_dict = {}
    stats = {}

    for i in range(len(video)):
        try:
            frame_rgb = video[i].asnumpy()
            video.seek(0)
            bounding_boxes, result_dict[i] = detect_holistic(frame_rgb)
        except Exception as e:
            print(f"Error on frame {i}: {e}")
            result_dict[i] = None
            continue

        stats[i] = bounding_boxes

    landmark_json_path = Path(pose_path) / f"{Path(video_file).stem}_pose.json"
    stats_json_path = Path(stats_path) / f"{Path(video_file).stem}_stats.json"

    with open(landmark_json_path, 'w') as f:
        json.dump(result_dict, f)
    with open(stats_json_path, 'w') as f:
        json.dump(stats, f)


In [ ]:
# === USER CONFIGURATION ===
index = 0
batch_size = 9000000
time_limit = 90000000000000000000000  # in seconds
files_list_path = "raw_files_list.pkl.gz"
problem_file_path = "problem_files.txt"
pose_path = Path("pose_hands_face_data/")
stats_path = Path("stats_hands_face_data/")
face_model_path = "models/face_landmarker.task"
hand_model_path = "models/hand_landmarker.task"

# === Load video list from list file ===
with gzip.open(files_list_path, "rb") as f:
    fixed_list = pickle.load(f)

# === Prepare folders and file ===
pose_path.mkdir(parents=True, exist_ok=True)
stats_path.mkdir(parents=True, exist_ok=True)
Path(problem_file_path).touch(exist_ok=True)

# === Divide into batches ===
video_batches = [fixed_list[i:i + batch_size] for i in range(0, len(fixed_list), batch_size)]

# === Initialize MediaPipe models ===
base_options_face = python.BaseOptions(model_asset_path=face_model_path)
options_face = vision.FaceLandmarkerOptions(base_options=base_options_face,
                                            output_face_blendshapes=True,
                                            output_facial_transformation_matrixes=True,
                                            num_faces=6)
face_detector = vision.FaceLandmarker.create_from_options(options_face)

base_options_hand = python.BaseOptions(model_asset_path=hand_model_path)
options_hand = vision.HandLandmarkerOptions(base_options=base_options_hand,
                                            num_hands=6,
                                            min_hand_detection_confidence=0.05)
hand_detector = vision.HandLandmarker.create_from_options(options_hand)

mp_holistic = mp.solutions.holistic.Holistic(min_detection_confidence=0.1)

# === Main processing loop ===
start_time = time.time()

for video_file in video_batches[index]:
    if time.time() - start_time > time_limit:
        print("⏰ Time limit reached. Stopping execution.")
        break

    landmark_json_path = pose_path / f"{Path(video_file).stem}_pose.json"
    stats_json_path = stats_path / f"{Path(video_file).stem}_stats.json"

    if landmark_json_path.exists() and stats_json_path.exists():
        continue
    elif is_string_in_file(problem_file_path, video_file):
        continue
    else:
        print(f"Processing: {video_file}")
        video_holistic(video_file, problem_file_path, pose_path, stats_path)

I0000 00:00:1752136085.880532  848806 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1752136086.058764  863111 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 550.144.03), renderer: NVIDIA A40/PCIe/SSE2
W0000 00:00:1752136086.059435  848806 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1752136086.069150  863117 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1752136086.084071  863128 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1752136086.090385  848806 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1752136086.180609  863166 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 NVIDI

Processing: /DATA5/ashishu23/SURGE/iSign-videos_v1.1/LI3wO3YkU1M--2.mp4


W0000 00:00:1752136086.560387  863257 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1752136086.561267  863260 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
/home/ashishu23/miniconda3/envs/feature_extraction/lib/python3.10/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Processing: /DATA5/ashishu23/SURGE/iSign-videos_v1.1/O47tiJVLIX4--85.mp4
Processing: /DATA5/ashishu23/SURGE/iSign-videos_v1.1/53xfDxkxvug--1.mp4


KeyboardInterrupt: 

In [6]:
import os

def count_files_in_folder(folder_path):
    return sum(1 for entry in os.listdir(folder_path)
               if os.path.isfile(os.path.join(folder_path, entry)))

# Example usage
folder = "pose_hands_face_data"
print(f"Number of files: {count_files_in_folder(folder)}")


Number of files: 1258


## Crop the Face

In [8]:
import cv2
import numpy as np
import os
import pickle
import gzip
from datetime import datetime
from pathlib import Path
import decord
import json
import time


In [9]:
def load_file(filename):
    with gzip.open(filename, "rb") as f:
        return pickle.load(f)

def is_string_in_file(file_path, target_string):
    try:
        with Path(file_path).open("r") as f:
            return any(target_string in line for line in f)
    except Exception as e:
        print(f"Error: {e}")
        return False


In [10]:
def resize_frame(frame, frame_size):
    return cv2.resize(frame, frame_size, interpolation=cv2.INTER_AREA) if frame is not None and frame.size > 0 else None

def crop_frame(image, bounding_box):
    x, y, w, h = bounding_box
    return image[y:y + h, x:x + w]

def get_bounding_box(landmarks, image_shape, scale_factor=1.2):
    ih, iw, _ = image_shape
    landmarks_px = np.array([(int(l[0] * iw), int(l[1] * ih)) for l in landmarks])
    center_x, center_y = np.mean(landmarks_px, axis=0, dtype=int)
    xb, yb, wb, hb = cv2.boundingRect(landmarks_px)
    box_size = max(wb, hb)
    half_size = box_size // 2
    x = center_x - half_size
    y = center_y - half_size
    w, h = box_size, box_size

    w_padding = int((scale_factor - 1) * w / 2)
    h_padding = int((scale_factor - 1) * h / 2)
    x -= w_padding
    y -= h_padding
    w += 2 * w_padding
    h += 2 * h_padding    
    return x, y, w, h

def adjust_bounding_box(bounding_box, image_shape):
    x, y, w, h = bounding_box
    ih, iw, _ = image_shape
    x = max(min(x, iw - w), 0)
    y = max(min(y, ih - h), 0)
    return x, y, w, h


In [11]:
def select_face(pose_landmarks, face_landmarks):
    nose_pose = pose_landmarks[0]
    nose_candidates = [face[0] for face in face_landmarks]
    closest_idx = np.argmin([np.linalg.norm(np.array(nose_pose) - np.array(n)) for n in nose_candidates])
    return face_landmarks[closest_idx]

def calculate_bounding_box(landmarks, indices, image_shape):
    x_coords = [landmarks[i][0] for i in indices]
    y_coords = [landmarks[i][1] for i in indices]
    return (
        int(min(x_coords) * image_shape[1]), int(min(y_coords) * image_shape[0]),
        int(max(x_coords) * image_shape[1]), int(max(y_coords) * image_shape[0])
    )


In [12]:
def cues_on_grey_background(image, facial_landmarks):
    shape = image.shape
    left_eye_idx = [69, 168, 156, 118, 54]
    right_eye_idx = [168, 299, 347, 336, 301]
    mouth_idx = [164, 212, 432, 18]
    
    left_box = calculate_bounding_box(facial_landmarks, left_eye_idx, shape)
    right_box = calculate_bounding_box(facial_landmarks, right_eye_idx, shape)
    mouth_box = calculate_bounding_box(facial_landmarks, mouth_idx, shape)
    
    min_x = max(0, min(left_box[0], right_box[0], mouth_box[0]) - 10)
    min_y = max(0, min(left_box[1], right_box[1], mouth_box[1]) - 10)
    max_x = min(shape[1], max(left_box[2], right_box[2], mouth_box[2]) + 10)
    max_y = min(shape[0], max(left_box[3], right_box[3], mouth_box[3]) + 10)

    side_len = max(max_x - min_x, max_y - min_y)
    grey_bg = np.ones((side_len, side_len, 3), dtype=np.uint8) * 128

    def crop_and_paste(src, dst, box, origin):
        x1, y1, x2, y2 = box
        dx, dy = origin
        crop = src[y1:y2, x1:x2]
        dst[dy:dy+crop.shape[0], dx:dx+crop.shape[1]] = crop

    crop_and_paste(image, grey_bg, left_box, (left_box[0]-min_x, left_box[1]-min_y))
    crop_and_paste(image, grey_bg, right_box, (right_box[0]-min_x, right_box[1]-min_y))
    crop_and_paste(image, grey_bg, mouth_box, (mouth_box[0]-min_x, mouth_box[1]-min_y))
    
    return grey_bg


In [13]:
def video_holistic(video_file, face_path, problem_file_path, pose_path):
    video = decord.VideoReader(video_file)
    fps = video.get_avg_fps()

    clip_path = Path(face_path) / f"{Path(video_file).stem}_face.mp4"
    json_path = Path(pose_path) / f"{Path(video_file).stem}_pose.json"
    clip_path.parent.mkdir(parents=True, exist_ok=True)
    
    if os.path.exists(clip_path):
        os.remove(clip_path)

    out_face = cv2.VideoWriter(str(clip_path), cv2.VideoWriter_fourcc(*'mp4v'), fps, (224, 224))

    with open(json_path, 'r') as f:
        result_dict = json.load(f)

    prev_face_frame = None
    prev_result_dict = None

    for i in range(len(video)):
        frame = video[i].asnumpy()
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        res = result_dict.get(str(i), None)

        if res is None:
            res = prev_result_dict
        else:
            prev_result_dict = res

        if res is None or res.get('pose_landmarks') is None:
            out_face.write(prev_face_frame if prev_face_frame is not None else np.zeros((224, 224, 3), dtype=np.uint8))
            continue

        if res.get('face_landmarks'):
            face_lmks = select_face(res['pose_landmarks'][0], res['face_landmarks'])
            face_frame = resize_frame(cues_on_grey_background(frame_rgb, face_lmks), (224, 224))
            out_face.write(face_frame)
            prev_face_frame = face_frame
        else:
            out_face.write(prev_face_frame if prev_face_frame is not None else np.zeros((224, 224, 3), dtype=np.uint8))

    out_face.release()


In [14]:
# === Parameters (manual override or load from file/UI) ===
index = 0
batch_size = 10
time_limit = 6000  # seconds

files_list_path = "raw_files_list.pkl.gz"
problem_file_path = "problem_files.txt"
pose_path = Path("pose_hands_face_data")
face_path = Path("face_video_data")

# === Load video list from list file ===
with gzip.open(files_list_path, "rb") as f:
    fixed_list = pickle.load(f)

# === Prepare folders and file ===
pose_path.mkdir(parents=True, exist_ok=True)
face_path.mkdir(parents=True, exist_ok=True)
Path(problem_file_path).touch()

video_batches = [fixed_list[i:i + batch_size] for i in range(0, len(fixed_list), batch_size)]

start_time = time.time()


In [15]:
for video_file in video_batches[index]:
    if time.time() - start_time > time_limit:
        print("Time limit reached.")
        break

    clip_path = face_path / f"{Path(video_file).stem}_face.mp4"
    if clip_path.exists():
        print(f"Skipping existing: {clip_path}")
        continue
    if is_string_in_file(problem_file_path, video_file):
        print(f"Problem file found for: {video_file}")
        continue

    try:
        video_holistic(video_file, face_path, problem_file_path, pose_path)
    except Exception as e:
        print(f"Error processing {video_file}: {e}")
        with open(problem_file_path, "a") as pf:
            pf.write(video_file + "\n")


## Crop the Hands

In [16]:
# Cell 1: Imports
import cv2
import numpy as np
import os
import pickle
import gzip
from datetime import datetime
from pathlib import Path
import decord
import argparse
import json
import time

In [17]:
# Cell 2: Utility Functions

def load_file(filename):
    with gzip.open(filename, "rb") as f:
        return pickle.load(f)

def is_string_in_file(file_path, target_string):
    try:
        with Path(file_path).open("r") as f:
            return any(target_string in line for line in f)
    except Exception as e:
        print(f"Error: {e}")
        return False

def resize_frame(frame, frame_size):
    if frame is not None and frame.size > 0:
        return cv2.resize(frame, frame_size, interpolation=cv2.INTER_AREA)
    return None

def crop_frame(image, bounding_box):
    x, y, w, h = bounding_box
    return image[y:y + h, x:x + w]

def get_bounding_box(landmarks, image_shape, scale_factor=1.2):
    ih, iw, _ = image_shape
    landmarks_px = np.array([(int(l[0] * iw), int(l[1] * ih)) for l in landmarks])
    center_x, center_y = np.mean(landmarks_px, axis=0, dtype=int)
    xb, yb, wb, hb = cv2.boundingRect(landmarks_px)
    box_size = max(wb, hb)
    half_size = box_size // 2
    x = center_x - half_size
    y = center_y - half_size
    w = h = box_size
    w_padding = int((scale_factor - 1) * w / 2)
    h_padding = int((scale_factor - 1) * h / 2)
    x -= w_padding
    y -= h_padding
    w += 2 * w_padding
    h += 2 * h_padding
    return x, y, w, h

def adjust_bounding_box(bounding_box, image_shape):
    x, y, w, h = bounding_box
    ih, iw, _ = image_shape
    if x + w > iw:
        x = iw - w
    if y + h > ih:
        y = ih - h
    x = max(x, 0)
    y = max(y, 0)
    return x, y, w, h

def select_hands(pose_landmarks, hand_landmarks, image_shape):
    if hand_landmarks is None:
        return None, None
    left_wrist_pose = pose_landmarks[15]
    right_wrist_pose = pose_landmarks[16]
    wrist_from_hand = [hand[0] for hand in hand_landmarks]
    left_hand_landmarks = right_hand_landmarks = None
    if right_wrist_pose is not None:
        min_dist = float('inf')
        for h in hand_landmarks:
            dist = np.linalg.norm(np.array(right_wrist_pose[:2]) - np.array(h[0][:2]))
            if dist < min_dist and dist < 0.1:
                min_dist = dist
                right_hand_landmarks = h
    if left_wrist_pose is not None:
        min_dist = float('inf')
        for h in hand_landmarks:
            dist = np.linalg.norm(np.array(left_wrist_pose[:2]) - np.array(h[0][:2]))
            if dist < min_dist and dist < 0.1:
                min_dist = dist
                left_hand_landmarks = h
    return left_hand_landmarks, right_hand_landmarks


In [18]:
# Cell 3: Core Function to Extract Hands from Video

def video_holistic(video_file, hand_path, problem_file_path, pose_path):
    video = decord.VideoReader(video_file)
    fps = video.get_avg_fps()

    base_name = Path(video_file).stem
    clip_hand1_path = Path(hand_path) / f"{base_name}_hand1.mp4"
    clip_hand2_path = Path(hand_path) / f"{base_name}_hand2.mp4"
    landmark_json_path = Path(pose_path) / f"{base_name}_pose.json"

    for f in [clip_hand1_path, clip_hand2_path]:
        if f.exists(): os.remove(f)

    out_hand1 = cv2.VideoWriter(str(clip_hand1_path), cv2.VideoWriter_fourcc(*'mp4v'), fps, (224, 224))
    out_hand2 = cv2.VideoWriter(str(clip_hand2_path), cv2.VideoWriter_fourcc(*'mp4v'), fps, (224, 224))

    with open(landmark_json_path, 'r') as f:
        result_dict = json.load(f)

    prev_hand1, prev_hand2, prev_result = None, None, None

    for i in range(len(video)):
        frame = video[i].asnumpy()
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = result_dict.get(str(i)) or prev_result
        if result is None:
            continue
        prev_result = result

        pose = result.get('pose_landmarks')
        hands = result.get('hand_landmarks')
        if pose is None:
            out_hand1.write(prev_hand1 if prev_hand1 is not None else np.zeros((224, 224, 3), np.uint8))
            out_hand2.write(prev_hand2 if prev_hand2 is not None else np.zeros((224, 224, 3), np.uint8))
            continue

        lh, rh = select_hands(pose[0], hands, frame_rgb.shape)
        result['left_hand_landmarks'] = lh
        result['right_hand_landmarks'] = rh

        for hand, out, prev in [
            (lh, out_hand1, prev_hand1),
            (rh, out_hand2, prev_hand2)
        ]:
            if hand is not None:
                box = get_bounding_box(hand, frame_rgb.shape, 1.5)
                box = adjust_bounding_box(box, frame_rgb.shape)
                hand_frame = crop_frame(frame_rgb, box)
                hand_frame = resize_frame(hand_frame, (224, 224))
                out.write(hand_frame)
                if out == out_hand1: prev_hand1 = hand_frame
                else: prev_hand2 = hand_frame
            else:
                out.write(prev if prev is not None else np.zeros((224, 224, 3), np.uint8))

    out_hand1.release()
    out_hand2.release()


In [19]:
index=0
batch_size=10
time_limit=36000
files_list="raw_files_list.pkl.gz"
problem_file_path="problem_files.txt"
pose_path="pose_hands_face_data"
hand_path="hands_video_data"


Path(hand_path).mkdir(parents=True, exist_ok=True)
fixed_list = load_file(files_list)

if not os.path.exists(problem_file_path):
    Path(problem_file_path).write_text("")

video_batches = [fixed_list[i:i + batch_size] for i in range(0, len(fixed_list), batch_size)]
start_time = time.time()

for video_file in video_batches[index]:
    if time.time() - start_time > time_limit:
        print("Time limit reached.")
        break

    clip_hand2 = Path(hand_path) / f"{Path(video_file).stem}_hand2.mp4"
    if clip_hand2.exists() or is_string_in_file(problem_file_path, video_file):
        continue

    try:
        video_holistic(video_file, hand_path, problem_file_path, pose_path)
    except Exception as e:
        print(f"Error with {video_file}: {e}")
        with open(problem_file_path, "a") as f:
            f.write(video_file + "\n")


## Create the list file of Poses

In [20]:
import os
import pickle
import gzip
from pathlib import Path

# === CONFIGURATION ===
input_dir = Path("pose_hands_face_data/")  # Change this to your JSON pose directory
output_path = Path("./")
output_path.mkdir(parents=True, exist_ok=True)

output_file = output_path / "pose_files_list.pkl.gz"  # Output filename
pose_exts = {'.json'}

# === SCAN POSE JSON FILES ===
pose_paths = [
    str(file.resolve()) for file in input_dir.rglob("*")
    if file.suffix.lower() in pose_exts
]

print(f"Found {len(pose_paths)} JSON pose files.")

# === SAVE TO .pkl.gz ===
with gzip.open(output_file, 'wb') as f:
    pickle.dump(pose_paths, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Saved pose file path list to: {output_file}")

Found 10 JSON pose files.
Saved pose file path list to: pose_files_list.pkl.gz


## Extract Body Features

In [21]:
import cv2
import numpy as np
import os
import pickle
import gzip
from datetime import datetime
from pathlib import Path
import decord
import json
import glob
import time

In [22]:
def get_mp4_files(directory):
    if not os.path.exists(directory):
        raise FileNotFoundError(f'Directory not found: {directory}')
    return [os.path.abspath(file) for file in glob.glob(os.path.join(directory, '*.mp4'))]

def load_file(filename):
    with gzip.open(filename, "rb") as f:
        return pickle.load(f)

def is_string_in_file(file_path, target_string):
    try:
        with Path(file_path).open("r") as f:
            return any(target_string in line for line in f)
    except Exception as e:
        print(f"Error: {e}")
        return False


In [23]:
def normalize_pose_keypoints(pose_landmarks):
    left_shoulder = np.array(pose_landmarks[11][:2])
    right_shoulder = np.array(pose_landmarks[12][:2])
    left_eye = np.array(pose_landmarks[2][:2])
    nose = np.array(pose_landmarks[0][:2])

    head_unit = np.linalg.norm(right_shoulder - left_shoulder) / 2
    signing_space_width = 6 * head_unit
    signing_space_height = 7 * head_unit

    signing_space_top = left_eye[1] - 0.5 * head_unit
    signing_space_left = nose[0] - signing_space_width / 2

    translation_matrix = np.array([[1, 0, -signing_space_left],
                                   [0, 1, -signing_space_top],
                                   [0, 0, 1]])
    scale_matrix = np.array([[1 / signing_space_width, 0, 0],
                             [0, 1 / signing_space_height, 0],
                             [0, 0, 1]])
    shift_matrix = np.array([[1, 0, -0.5],
                             [0, 1, -0.5],
                             [0, 0, 1]])
    transformation_matrix = shift_matrix @ scale_matrix @ translation_matrix

    normalized_keypoints = []
    for landmark in pose_landmarks:
        keypoint = np.array([landmark[0], landmark[1], 1])
        normalized_keypoint = transformation_matrix @ keypoint
        normalized_keypoints.append(normalized_keypoint[:2])
    return normalized_keypoints


In [24]:
def keypoints_to_numpy(pose_file, pose_emb_path):
    try:
        with open(pose_file, 'r') as rd:
            result_dict = json.load(rd)
    except Exception as e:
        print(f"Failed to read {pose_file}: {e}")
        return

    prev_pose = None
    video_pose_landmarks = []

    for i in range(len(result_dict)):
        frame_data = result_dict.get(str(i))
        if frame_data is None:
            frame_pose_landmarks = prev_pose if prev_pose is not None else np.full((7, 2), -9999)
        elif frame_data['pose_landmarks'] is not None:
            frame_pose_landmarks = frame_data['pose_landmarks'][0]
            frame_pose_landmarks = normalize_pose_keypoints(frame_pose_landmarks[0:25])
            indices = [0, 11, 12, 13, 14, 15, 16]
            frame_pose_landmarks = np.array([frame_pose_landmarks[j] for j in indices]).flatten()
            prev_pose = frame_pose_landmarks
        else:
            frame_pose_landmarks = prev_pose if prev_pose is not None else np.full((7, 2), -9999).flatten()
        video_pose_landmarks.append(frame_pose_landmarks)

    video_pose_landmarks = np.array(video_pose_landmarks)
    video_pose_landmarks[:, :2] = -9999.0  # mask nose
    np_path = Path(pose_emb_path) / f"{Path(pose_file).stem}.npy"
    np_path.parent.mkdir(parents=True, exist_ok=True)
    np.save(np_path, video_pose_landmarks)


In [25]:
index=0
files_list="pose_files_list.pkl.gz"
pose_features_path="body_pose_data/"
batch_size=100
time_limit=36000

start_time = time.time()
fixed_list = load_file(files_list)
video_batches = [fixed_list[i:i + batch_size] for i in range(0, len(fixed_list), batch_size)]

for pose_file in video_batches[index]:
    pose_file = Path(pose_file)
    np_path = Path(pose_features_path) / f"{pose_file.stem}.npy"

    if np_path.exists():
        continue

    if time.time() - start_time > time_limit:
        print("Time limit reached. Stopping execution.")
        break

    keypoints_to_numpy(pose_file, pose_features_path)
